# SPINE-GPE v7 — Certificação PNADc Plataformas 2022/2024
Executa auditoria de layouts, leitura fixed-width, desenho survey, certified tables e golden tests SIDRA.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7')
SCRIPT = Path('/content/SPINE_GPEv7_PNADC_CERTIFIER_v1.0.0.py')
REQ = Path('/content/requirements_SPINE_GPEv7_PNADC_CERTIFIER_v1.0.0.txt')
print('ROOT:', ROOT, ROOT.exists())
print('SCRIPT:', SCRIPT, SCRIPT.exists())


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)], check=True)


## 1. Auditoria rápida
Não lê os microdados completos.

In [ ]:
audit = subprocess.run([
    sys.executable, str(SCRIPT),
    '--root', str(ROOT),
    '--mode', 'audit',
    '--strict',
], check=False)
print('Audit exit code:', audit.returncode)


## 2. Certificação completa
A leitura ocorre em chunks. Os TXT são copiados para `/content` para reduzir o gargalo do Drive.

In [ ]:
cert = subprocess.run([
    sys.executable, str(SCRIPT),
    '--root', str(ROOT),
    '--mode', 'certify',
    '--chunk-rows', '20000',
    '--strict',
], check=False)
print('Certification exit code:', cert.returncode)


## 3. Abrir lock e relatório

In [ ]:
import json
LOCK = ROOT / '00_admin' / 'PNADC_CERTIFICATION_LOCK.json'
REPORT = ROOT / '06_reports' / 'pnadc_certification' / 'pnadc_certification_report_LATEST.md'
print(json.dumps(json.loads(LOCK.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))
print('\n' + REPORT.read_text(encoding='utf-8')[-20000:])


## 4. Inspecionar certified tables

In [ ]:
import pandas as pd
for year in (2022, 2024):
    path = ROOT / '03_processed' / '10_pnadc_certified' / f'certified_pnadc_platform_{year}.parquet'
    df = pd.read_parquet(path, columns=['source_year','reference_quarter','SD14001','S140093','platform_any_direct','platform_delivery_direct','survey_weight'])
    print(year, df.shape)
    display(df.head())
